# 第8章 货币市场工具与回购市场 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch08_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch08_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 6：例8.1/8.2 + ROE-杠杆曲线（正/负 carry）


In [ ]:
import numpy as np
from fi import repo, plotting
plotting.use_chinese_style()
cf = repo.repo_cashflows(1e8, 0.0185, 7)
print('例8.1 利息=', round(cf['interest'],2))
for L in (1,3): print(f'例8.2 L={L}: ROE={repo.leveraged_carry(0.0255,0.0185,L)["roe_annual"]*100:.2f}%')
Ls = np.linspace(1,5,81)
fig, ax = plotting.new_axes()
ax.plot(Ls, [repo.leveraged_carry(0.0255,0.0185,L)['roe_annual']*100 for L in Ls], label='r=1.85%(正carry)')
ax.plot(Ls, [repo.leveraged_carry(0.0255,0.0280,L)['roe_annual']*100 for L in Ls], label='r=2.80%(负carry)')
ax.axhline(2.55, ls=':', color='gray', label='不加杠杆票息')
ax.set_xlabel('杠杆 L'); ax.set_ylabel('ROE (%)'); ax.set_title('正/负 carry 下 ROE 随杠杆'); ax.legend()
fig.tight_layout()


## 编程实验 7：money_market 样本复盘（carry 时序 + L=1 vs L=3）


In [ ]:
import pandas as pd
from fi import data
mm = data.load_sample('money_market'); mm['date'] = pd.to_datetime(mm['date']); mm = mm.set_index('date')
carry = mm['cgb_10y'] - mm['dr007']
y_d, r_d = mm['cgb_10y']/100, mm['dr007']/100
def daily(L): return (y_d + (L-1)*(y_d-r_d))/250
cum1, cum3 = (1+daily(1)).cumprod(), (1+daily(3)).cumprod()
mdd = lambda nav: float(((nav/nav.cummax())-1).min())
print(f'L=1 期末={cum1.iloc[-1]:.4f} 回撤={mdd(cum1)*100:.3f}%')
print(f'L=3 期末={cum3.iloc[-1]:.4f} 回撤={mdd(cum3)*100:.3f}%  (样本carry恒正故无回撤)')
fig, ax = plotting.new_axes()
ax.plot(cum1.index, cum1.values, label='L=1'); ax.plot(cum3.index, cum3.values, label='L=3')
ax.set_ylabel('累计净值'); ax.set_title('杠杆前后累计回报'); ax.legend(); fig.tight_layout()


## 编程实验 8：akshare 真实数据复盘（联网）


In [ ]:
print('联网示意：')
print('import akshare as ak')
print('dr = ak.rate_interbank(market="上海银行间同业拆放利率")  # 或 DR007 相关接口')
print('取真实 DR007/R007/10Y 国债，按上面方法算 carry 与杠杆回报，')
print('并标注 R007-DR007 利差最大的交易日，结合新闻讨论流动性事件（如季末、钱荒）')
